# Anima 生图 —— **Kaggle 版**（T4 x2，免费 GPU 额度）

由 `examples/anima_colab.ipynb`（v7.5）适配。已实测前提：
**animagine-xl-4.0 fp16 / 832×1216 / 28 步 = 22.1s/张**（mean 161.8，md5 `1bb81b4d…`）；
**cloudflared quick tunnel 在 Kaggle 出网白名单内**（探针成功，location=iad21）。

## 开跑前（必须手动，脚本不能代劳）
1. 右上 **Settings → Accelerator = `GPU T4 x2`**（或 P100）
2. 同面板 **Internet = `On`**
3. Kaggle 账号需**已完成手机验证**，否则 GPU 不可选

## ⚠️ 必须 interactive 跑，不要 "Save & Run All"
- tunnel URL 是 cell 4 **运行时**打印的，提交式运行拿不到实时 URL
- Kaggle kernel **空闲会自动停**，一停隧道立刻断
- 定位：**手动出图环境**，不是后台挂机服务

## 跑完存 URL（两个文件名等价）
```
echo 'https://xxx.trycloudflare.com' > ~/.dsh/.colab-url
echo 'https://xxx.trycloudflare.com' > ~/.dsh/.kaggle-url
```

## 与 Colab 版的差别（本轮有意不移植）
| 项 | Kaggle 版 | 原因 |
|---|---|---|
| sdcpp / GGUF CPU 引擎 | 不移植 | Kaggle 有 GPU；CPU 上 SDXL 实测 160s/步 |
| 视频 `kind=video` | 不移植 | 权重 10–20GB，先把出图跑稳 |
| Drive 落盘 | 跳过 | 无 Drive 挂载，产物落 `/kaggle/working/out` |


In [ ]:
#@title 1. 环境探测 + 依赖（先跑这格）
import os, sys, subprocess, shutil
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')  # 抗显存碎片

IN_COLAB = os.path.isdir('/content')
IN_KAGGLE = os.path.isdir('/kaggle')
WORK = '/kaggle/working' if IN_KAGGLE else ('/content' if IN_COLAB else os.path.abspath('.'))
print('colab:', IN_COLAB, '| kaggle:', IN_KAGGLE, '| workdir:', WORK)

HAS_GPU = False; GPU_NAME = 'none'; VRAM_GB = 0.0
try:
    r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                       capture_output=True, text=True, timeout=30)
    if r.returncode == 0 and r.stdout.strip():
        GPU_NAME = r.stdout.split(',')[0].strip()
        VRAM_GB = float(r.stdout.split(',')[1].replace('MiB','').strip()) / 1024
        HAS_GPU = True
except Exception as e:
    print('nvidia-smi:', e)
print('GPU : %s %.1fGB' % (GPU_NAME, VRAM_GB) if HAS_GPU else 'GPU : 无（CPU 档 -> OpenVINO 生图）')

try:
    import psutil
    vm = psutil.virtual_memory()
    print('RAM : %.1f 总 / %.1f 空闲' % (vm.total/1024**3, vm.available/1024**3))
except ImportError:
    pass
print('disk: %.1fGB free' % (shutil.disk_usage('/content' if IN_COLAB else '.').free/1024**3))

# ---- CPU 能力探测：bf16 方案成不成立全看这里 ----
CPU_FLAGS = set(); CPU_MODEL = 'unknown'
try:
    for _l in open('/proc/cpuinfo'):
        if _l.startswith('model name') and CPU_MODEL == 'unknown':
            CPU_MODEL = _l.split(':', 1)[1].strip()
        if _l.startswith('flags') or _l.startswith('Features'):
            CPU_FLAGS |= set(_l.split(':', 1)[1].split())
except Exception as e:
    print('cpuinfo:', e)
VCPU = os.cpu_count() or 2
CPU_BF16_HW = ('avx512_bf16' in CPU_FLAGS) or ('amx_bf16' in CPU_FLAGS)
print('CPU : %s | vCPU %s | bf16硬件加速=%s' % (CPU_MODEL, VCPU, CPU_BF16_HW))
print('      AVX512/AMX 位:', ' '.join(sorted(x for x in CPU_FLAGS if 'avx512' in x or 'amx' in x))
      or '无 → CPU 上 bfloat16 会走软件 fallback，不比 fp32 快，方案一不适用')

# Kaggle 预装 torch 2.11.0+cu128；diffusers/transformers 要够新才认 animagine-xl-4.0 与 dtype=
# 装失败别硬撑：记错误原文，cell 3 改 GPU_MODEL='sd_turbo' 先跑通
if IN_COLAB or IN_KAGGLE:
    %pip -q install -U "diffusers>=0.35" "transformers>=4.44,<5" accelerate safetensors flask flask-cors psutil pillow requests numpy
    # CPU 档现在走纯 diffusers fp16，不再需要 optimum-intel/openvino（旧路已被证明装不上）。
    # 想再试 OpenVINO 加速：把下面注释打开（注意 optimum.intel 的无门 import 会拉 open_clip/kokoro/timm）
    # if not HAS_GPU:
    #     %pip -q install optimum-intel openvino open_clip_torch timm sentence_transformers kokoro

try:
    import diffusers, transformers
    print('[i] diffusers', diffusers.__version__, '| transformers', transformers.__version__)
except Exception as _e:
    print('[!] 依赖导入失败（原文）:', type(_e).__name__, _e)

import torch
print('[i] torch', torch.__version__, '| cuda', torch.version.cuda, '| available', torch.cuda.is_available())
if HAS_GPU:
    torch.backends.cuda.matmul.allow_tf32 = True   # T4 fp32 免费提速 ~2x
    torch.backends.cudnn.allow_tf32 = True
print('[+] 就绪 | torch', torch.__version__)

In [ ]:
#@title 2. 会话保活 + Drive 落盘
import os, time, shutil, threading

OUT_DIR = ('/kaggle/working/out' if IN_KAGGLE else
           ('/content/out' if IN_COLAB else os.path.abspath('./out')))
os.makedirs(OUT_DIR, exist_ok=True)

if IN_COLAB:
    from IPython.display import display, Javascript
    display(Javascript("""
    function ka(){var b=document.querySelector('colab-connect-button');
      if(b&&b.shadowRoot){var c=b.shadowRoot.querySelector('#connect');if(c)c.click();}}
    setInterval(ka,60000);"""))
    print('[+] keepalive 60s')

DRIVE_OK = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/ai_out'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    DRIVE_OK = True
    print('[+] Drive ->', DRIVE_DIR)
except Exception as e:
    print('[!] 无 Drive（用浏览器下载也行）:', str(e)[:120])

def flush_to_drive(verbose=True):
    if not DRIVE_OK:
        if verbose: print('[!] Drive 不可用，产出在', OUT_DIR)
        return
    tgt = os.path.join(DRIVE_DIR, time.strftime('%m%d_%H%M'))
    os.makedirs(tgt, exist_ok=True)
    n = 0
    for root, _, files in os.walk(OUT_DIR):
        for f in files:
            shutil.copy2(os.path.join(root, f), os.path.join(tgt, f)); n += 1
    if verbose: print('[+] 搬了 %d 个文件 -> %s' % (n, tgt))

In [ ]:
#@title 3. 加载 GPU 模型（默认 animagine-xl-4.0 fp16，实测 1024@28步 ~32s）
# Anima 的硬件墙：bf16 训练的模型，T4 无 bf16 -> fp16 溢出黑图，只能 fp32(慢)。
# 所以 GPU 档默认换 SDXL 动漫底（fp16 原生安全）。改 GPU_MODEL 切换：
#   'animagine'  SDXL 动漫天花板，fp16 28步 ~30s/张          （推荐，已实测）
#   'nova_lcm'   SDXL 动漫 LCM 蒸馏，8步 ~10s/张（画质略降）
#   'anima'      Anima fp32，512x768@16 ~64s（那种质感要它，慢）
#   'sd_turbo'   SD1.5 蒸馏，512x768@4 极快（**只用于测速/兜底**，画质明显低于 SDXL）
GPU_MODEL = 'animagine'
ANIMA_ID = 'circlestone-labs/Anima-Base-v1.0-Diffusers'
SDXL_IDS = {'animagine': 'cagliostrolab/animagine-xl-4.0', 'nova_lcm': 'sca255/nova_xl_lcm'}
SD15_IDS = {'sd_turbo': 'stabilityai/sd-turbo'}   # 非 SDXL：走 StableDiffusionPipeline
GPU_DEFAULTS = {'animagine': {'w': 832, 'h': 1216, 'steps': 28, 'guidance': 7.0},
                'nova_lcm':  {'w': 1024, 'h': 1024, 'steps': 8,  'guidance': 1.5},
                'anima':     {'w': 512,  'h': 768,  'steps': 16, 'guidance': None},
                'sd_turbo':  {'w': 512,  'h': 768,  'steps': 4,  'guidance': 0.0}}
pipe = None

if not HAS_GPU:
    print('[!] CPU 档：本格跳过（cell 4 自动走 OpenVINO 生图）')
else:
    import torch, gc, numpy as np

    def _fp16(cls, mid, **kw):
        try:
            return cls.from_pretrained(mid, dtype=torch.float16, **kw)
        except TypeError:
            return cls.from_pretrained(mid, torch_dtype=torch.float16, **kw)

    def _load(name):
        global pipe
        if pipe is not None:
            del pipe
        gc.collect(); torch.cuda.empty_cache()
        if name == 'anima':
            from diffusers import ModularPipeline
            p = _fp16(ModularPipeline, ANIMA_ID)
            p.load_components()
            p.to('cuda')
            for a in ('text_encoder', 'text_conditioner', 'transformer', 'vae'):
                c = getattr(p, a, None)
                if c is not None:
                    try:
                        c.to(torch.float32)   # GPU 上原地升 fp32，CPU 里永远只有 fp16 一份
                    except Exception:
                        pass
        elif name in SD15_IDS:
            # SD1.5 蒸馏档：只用来测速/兜底，画质不参与选型
            from diffusers import StableDiffusionPipeline
            p = _fp16(StableDiffusionPipeline, SD15_IDS[name], use_safetensors=True)
            p.to('cuda')
        else:
            from diffusers import StableDiffusionXLPipeline
            p = _fp16(StableDiffusionXLPipeline, SDXL_IDS[name], use_safetensors=True)
            p.to('cuda')
        p.set_progress_bar_config(disable=True)
        try:
            p.enable_xformers_memory_efficient_attention()
        except Exception:
            try:
                p.enable_attention_slicing()
            except Exception:
                pass
        gc.collect(); torch.cuda.empty_cache()
        return p

    def _test(name):
        cfg = GPU_DEFAULTS[name]
        kw = dict(height=512, width=512, num_inference_steps=8,
                  generator=torch.Generator('cuda').manual_seed(7))
        if cfg['guidance'] is not None:
            kw['guidance_scale'] = cfg['guidance']
        out = pipe(prompt='masterpiece, best quality, 1girl, portrait',
                   negative_prompt='worst quality', **kw)
        m = float(np.asarray(out.images[0].convert('RGB')).mean())
        del out
        gc.collect(); torch.cuda.empty_cache()
        print('  [%s] 测试图 mean=%.1f -> %s' % (name, m, '真图' if m >= 6 else '黑图'))
        return m >= 6

    try:
        pipe = _load(GPU_MODEL)
        if _test(GPU_MODEL):
            free_b, total_b = torch.cuda.mem_get_info()
            c = GPU_DEFAULTS[GPU_MODEL]
            print('[+] GPU_MODEL=%s 就绪 | 显存空闲 %.1f/%.1f GB | 默认 %dx%d@%d步'
                  % (GPU_MODEL, free_b/1024**3, total_b/1024**3, c['w'], c['h'], c['steps']))
        else:
            print('[x] %s 黑图/异常 -> 改 GPU_MODEL 为其它档重跑本格' % GPU_MODEL)
            pipe = None
    except Exception as e:
        print('[x] cell3 异常: %s: %s' % (type(e).__name__, str(e)[:300]))

In [ ]:
#@title 4. 启动服务 + 隧道（DSH / CLI 直调入口）
import threading, uuid, os, io, json, time, re, queue, subprocess, urllib.request, shutil, sys
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
import numpy as np

BUILD = 'kg1'   # Kaggle 版自证版本号（/health 回显）
# cell 1 定义的 WORK/IN_KAGGLE 在这里兜底：cell 4 单独重跑也不会 NameError
WORK = globals().get('WORK') or ('/kaggle/working' if os.path.isdir('/kaggle')
                                 else ('/content' if os.path.isdir('/content') else os.path.abspath('.')))
IN_KAGGLE = globals().get('IN_KAGGLE', os.path.isdir('/kaggle'))   # 每次改 cell 4 的逻辑就 +1；/health 会回显，用于确认 runtime 跑的是哪版
app = Flask(__name__); CORS(app)
JOBS = {}; LOCK = threading.Lock(); TASK_Q = queue.Queue()
URL = None

# ---- CPU 档预设（无 GPU 才用）----
# 2026-10-18 重写：彻底不碰 optimum-intel / OpenVINO。
# 根因（静态分析 optimum-intel 2.2.0 wheel 坐实）：
#   optimum/intel/openvino/__init__.py  L84 `from .modeling_open_clip import ...`
#                                       L97 `from .modeling_text2speech import ...`
#   这两行**没有 is_xxx_available() 门** → import optimum.intel 就硬拉
#   open_clip / kokoro / timm / sentence_transformers，Colab 镜像一个都没装
#   → 底层 ModuleNotFoundError 被 transformers._LazyModule 重写成
#     "Could not import module 'OVDiffusionPipeline'. Are this object's requirements defined correctly?"
#   与 diffusers / transformers / openvino 版本无关，换任何 cpu_preset 都炸 → 旧 CPU 档整条路死。
# 新方案：纯 diffusers fp16 + enable_model_cpu_offload（RAM 峰值只多一个组件）。
#   engine 可选：'diffusers'(默认稳) | 'openvino'(显式尝试，失败自动降级)
CPU_ENGINE = 'diffusers'
CPU_SLICING = False   # True = 开 attention/vae slicing（更省内存但明显更慢）
CPU_SDPA = True         # 给 unet/transformer 挂 SDPA(native)：CPU 上 bf16/fp32 都受益且无副作用
CPU_COMPILE = False     # torch.compile 首次编译在 Colab CPU 要数分钟、隧道易超时 → 默认关
CPU_BF16_OVERRIDE = None  # None=按 /proc/cpuinfo 探测；True/False 强制（对照实验用）
# ram_gb: pipeline 权重实测体积（HF blobs API，只算子目录 safetensors，不含根目录 single-file ckpt）。
#   只有 fp16 权重的仓库（sdxl_anime/nova_xl_lcm）要 fp32 就得下载后转精度 → 峰值翻倍，故 float32 预算按 2x 填。
#   超闸门就不加载，避免把 Colab 会话打死（12.6GB RAM 墙）。
CPU_PRESETS = {
    'sd_turbo':    {'cls':'StableDiffusionPipeline','model':'stabilityai/sd-turbo','steps':4,'guidance':0.0,'maxdim':768,'neg':False,'variant':'fp16','extra':{},'ram_gb':{'bfloat16':2.6,'float16':2.6,'float32':5.2},'act_gb':2.5},
    'sdxl_anime':  {'cls':'StableDiffusionXLPipeline','model':'sca255/nova_xl_lcm','steps':8,'guidance':1.5,'maxdim':1024,'neg':True,'variant':'fp16','extra':{},'ram_gb':{'bfloat16':7.4,'float16':7.4,'float32':14.7},'act_gb':4.0},
    'sdxl_light':  {'cls':'StableDiffusionXLPipeline','model':'stabilityai/sdxl-turbo','steps':4,'guidance':0.0,'maxdim':768,'neg':False,'variant':'fp16','extra':{},'ram_gb':{'bfloat16':6.9,'float16':6.9,'float32':13.9},'act_gb':3.5},
    'animagine':   {'cls':'StableDiffusionXLPipeline','model':'cagliostrolab/animagine-xl-4.0','steps':25,'guidance':7.0,'maxdim':1024,'neg':True,'variant':None,'extra':{},'ram_gb':{'bfloat16':6.9,'float16':6.9,'float32':6.9},'act_gb':4.5},
    'sana06':      {'cls':'SanaSprintPipeline','model':'Efficient-Large-Model/Sana_Sprint_0.6B_1024px_diffusers','steps':4,'guidance':4.5,'maxdim':1024,'neg':False,'variant':None,'extra':{'intermediate_timesteps':None,'clean_caption':True},'ram_gb':{'bfloat16':7.7,'float16':7.7,'float32':7.7},'act_gb':3.0},
}
CPU_PRESET = 'sdxl_anime'
CPU = {'pipe': None, 'key': None, 'loading': False, 'error': None, 'engine': None, 'dtype': None}

def set_cpu_preset(name):
    global CPU_PRESET
    if name in CPU_PRESETS and name != CPU_PRESET:
        CPU_PRESET = name; CPU['pipe'] = None; CPU['key'] = None; CPU['error'] = None

def _free_ram(tag=''):
    import gc
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    try:
        import psutil
        vm = psutil.virtual_memory()
        print('[i] RAM%s %.1f 总 / %.1f 空闲' % (tag, vm.total/1024**3, vm.available/1024**3))
    except Exception:
        pass

def _cpu_bf16_capable():
    if CPU_BF16_OVERRIDE is not None:
        return CPU_BF16_OVERRIDE
    f = globals().get('CPU_FLAGS')
    if f is None:
        f = set()
        try:
            for _l in open('/proc/cpuinfo'):
                if _l.startswith('flags') or _l.startswith('Features'):
                    f |= set(_l.split(':', 1)[1].split()); break
        except Exception:
            pass
    return ('avx512_bf16' in f) or ('amx_bf16' in f)

def _avail_ram_gb():
    try:
        import psutil
        return psutil.virtual_memory().available / 1024**3
    except Exception:
        return 99.0

def _load_cpu_diffusers(pr, _w=None, _h=None):
    import torch
    from diffusers import StableDiffusionPipeline, StableDiffusionXLPipeline
    try:
        from diffusers import SanaSprintPipeline
    except ImportError:
        SanaSprintPipeline = None
    CLS = {'StableDiffusionPipeline': StableDiffusionPipeline,
           'StableDiffusionXLPipeline': StableDiffusionXLPipeline,
           'SanaSprintPipeline': SanaSprintPipeline}[pr['cls']]
    if CLS is None:
        raise RuntimeError('diffusers 无 %s，换 cpu_preset' % pr['cls'])
    kw = {'use_safetensors': True}
    pr_variant = pr.get('variant')
    # ★ CPU 上 fp32 才是快路径：PyTorch 的 fp16 matmul 在 CPU 上是逐算子升 fp32 模拟的，
    #   实测比 fp32 慢数倍（sd_turbo 4步512px 用 fp16 跑了 20+ 分钟没出图）。
    #   所以顺序是 fp32 优先；只有 fp32 过不了 RAM 闸门才退 fp16（省一半内存、换速度）。
    #   有 CUDA 时反过来：fp16 又快又省。
    try:
        _has_cuda = torch.cuda.is_available()
    except Exception:
        _has_cuda = False
    _bf = _cpu_bf16_capable()
    if _has_cuda:
        order = ('float16', 'bfloat16', 'float32')
    elif _bf:
        # CPU 有 bf16 硬件：bf16 权重仅 7.3GB（fp32 的 14.7GB 装不进 12.6GB 墙）
        # → 唯一"装得下 + 可能快"的路，直接破 SDXL 两难
        order = ('bfloat16', 'float32', 'float16')
    else:
        # 无 bf16 硬件：bf16 软件 fallback 反而慢，跳过
        order = ('float32', 'float16')
    # force_dtype: 用户显式指定精度（闸门仍生效，但不再"自动退回慢速 fp16"）
    _force = str(CPU.get('force_dtype') or '').strip()
    if _force in ('float32', 'float16', 'bfloat16'):
        order = (_force,)
        print('[i] force_dtype=%s：跳过自动 dtype 选择' % _force)
    last = None
    wbase = pr.get('ram_gb', {})
    act = pr.get('act_gb', 3.0)
    # 闸门必须是**分辨率感知**的：act_gb 是按 1024² 估的，512² 只要约 1/4。
    # v6 用固定 act_gb → 出小图也被拦（2026-10-18 实测 sana06 1024 被拦后退了 fp16，
    # fp16 在 CPU 上 588s/步，等于把可用任务判死）。
    try:
        _mp = (float(_w or 1024) * float(_h or 1024)) / (1024.0 * 1024.0)
    except Exception:
        _mp = 1.0
    act = act * max(0.55, min(1.0, _mp))
    # ★ 权重之外还要 **1.5 倍瞬时峰值**：diffusers 的 from_pretrained 是"读 safetensors → 再拷一份到模块"，
    #   最大的那个组件（unet）会瞬时双份。2026-10-18 真机坐实：
    #   animagine fp32（权重 6.9GB）在 avail 11.6GB 的 runtime 上**过了旧闸门**，
    #   avail 一路 11.6 → 9.6 → 5.8 之后 kernel 被打死（/health 永久 530）。
    #   sd_turbo（5.2GB）在同一环境能活，sana06（7.7GB）在 768² 上也死过 → 1.5x 系数同时解释三者。
    print('[i] RAM 闸门: %.2fMP -> act %.1fGB | 可用 %.1fGB' % (_mp, act, _avail_ram_gb()))
    for dt in order:
        need = wbase.get(dt)
        if need:
            peak = need * 1.5 + (act if dt == 'float32' else act * 0.6)   # 1.5x = 加载瞬时双份
            if _avail_ram_gb() < peak + 0.8:
                last = RuntimeError('%s 需 ~%.1fGB(权重%.1f+推理峰值%.1f) RAM，当前只剩 %.1fGB -> 换更小的 cpu_preset（sd_turbo 权重仅 2.6GB）或降低分辨率'
                                    % (dt, peak, need, (act if dt=='float32' else round(act*0.6,1)), _avail_ram_gb()))
                print('[x]', str(last), '-> 试下一个 dtype')
                continue
        kw2 = dict(kw)
        if pr_variant and dt in ('float16', 'bfloat16'):
            kw2['variant'] = pr_variant
        try:
            try:
                p = CLS.from_pretrained(pr['model'], dtype=getattr(torch, dt), **kw2)
            except TypeError:        # 老版 diffusers 只认 torch_dtype
                p = CLS.from_pretrained(pr['model'], torch_dtype=getattr(torch, dt), **kw)
            print('[+] CPU 权重 dtype=%s' % dt)
            CPU['dtype'] = dt
            return p
        except Exception as e:
            last = e
            msg = str(e)
            if kw2.get('variant') and ('fp16' in msg or 'not found' in msg or 'safetensors' in msg):
                print('[!] variant=fp16 不可用(%s) -> 去掉 variant 重试' % msg[:120])
                pr_variant = None
                continue
            _free_ram(' (重载前)')
    raise last

# （Kaggle 版不移植 sdcpp/GGUF：有 GPU，CPU 上 SDXL 实测 160s/步，无意义）
SDCPP_DIR = WORK   # /sh 与上传等辅助路径统一落工作目录

def _load_cpu_openvino(pr):
    from optimum.intel import OVDiffusionPipeline
    return OVDiffusionPipeline.from_pretrained(pr['model'], compile=True)

def load_cpu_pipe(_w=None, _h=None):
    if CPU['pipe'] is not None and CPU['key'] == CPU_PRESET:
        return CPU['pipe']
    if CPU['loading']:
        raise RuntimeError('CPU model loading, retry in 120s')
    CPU['loading'] = True; CPU['error'] = None
    pr = CPU_PRESETS[CPU_PRESET]
    try:
        if CPU['pipe'] is not None:
            del CPU['pipe']; CPU['pipe'] = None; CPU['key'] = None
            _free_ram(' (切档)')
        p = None
        if CPU_ENGINE == 'openvino':
            try:
                p = _load_cpu_openvino(pr); CPU['engine'] = 'openvino'; CPU['dtype'] = 'ov'
            except Exception as e:
                print('[!] OpenVINO 档失败(%s: %s) -> 降级 diffusers fp16'
                      % (type(e).__name__, str(e)[:160]))
                p = None
        if p is None:
            p = _load_cpu_diffusers(pr, _w, _h); CPU['engine'] = 'diffusers-' + (CPU.get('dtype') or 'fp32')
        p.set_progress_bar_config(disable=True)
        # 纯 CPU runtime 没有 accelerator，offload 会抛 ValueError；只有真 CUDA 时才挂 offload
        try:
            import torch as _t
            _accel = _t.cuda.is_available()
        except Exception:
            _accel = False
        if _accel:
            try:
                p.enable_model_cpu_offload(); print('[+] offload: model_cpu (权重常驻 RAM，逐组件上卡)')
            except Exception:
                try:
                    p.enable_sequential_cpu_offload(); print('[+] offload: sequential_cpu')
                except Exception:
                    print('[!] offload 全失败，按纯 CPU 跑')
        else:
            try:
                p.to('cpu')
            except Exception as e:
                print('[!] .to(cpu) 失败: %s' % str(e)[:120])
            # 纯 CPU：attention slicing / vae tiling 省的是显存，CPU 上只会拖慢，默认不开。
            # 真要省内存再手动打开（CPU_SLICING=1）。
            if CPU_SLICING:
                try:
                    p.enable_attention_slicing()
                    p.enable_vae_slicing(); p.enable_vae_tiling()
                    print('[i] CPU_SLICING=1 已开 attention/vae slicing（慢但省内存）')
                except Exception:
                    pass
            else:
                print('[i] 纯 CPU：不开 slicing（省显存无用、纯减速）')
            try:
                import torch as _t
                _n = _t.get_num_threads()
                _avail = os.cpu_count() or 2
                if _n < _avail:
                    _t.set_num_threads(_avail)
                print('[i] torch 线程 %d -> %d (cpu_count=%s)' % (_n, _t.get_num_threads(), _avail))
            except Exception as e:
                print('[!] set_num_threads 失败: %s' % str(e)[:100])
        if CPU_SDPA:
            try:
                for _at in ('unet', 'transformer'):
                    _m = getattr(p, _at, None)
                    if _m is not None and hasattr(_m, 'set_attention_backend'):
                        _m.set_attention_backend('native'); print('[+] SDPA(native) 挂到 %s' % _at)
            except Exception as e:
                print('[!] SDPA 设置失败(不影响出图): %s' % str(e)[:100])
        if CPU_COMPILE:
            try:
                for _at in ('unet', 'transformer'):
                    _m = getattr(p, _at, None)
                    if _m is not None:
                        setattr(p, _at, torch.compile(_m))
                        print('[+] torch.compile(%s) 已挂（首次调用要编译数分钟）' % _at)
            except Exception as e:
                print('[!] torch.compile 失败(不影响出图): %s' % str(e)[:100])
        _free_ram(' (加载后)')
        CPU['pipe'] = p; CPU['key'] = CPU_PRESET
        print('[+] CPU 档就绪 | preset=%s engine=%s %s 默认 %d步 cfg%.1f <=%dpx'
              % (CPU_PRESET, CPU['engine'], pr['model'], pr['steps'], pr['guidance'], pr['maxdim']))
        return p
    except Exception as e:
        CPU['error'] = ('%s: %s' % (type(e).__name__, str(e)))[:300]
        raise
    finally:
        CPU['loading'] = False

def set_job(jid, **kw):
    with LOCK:
        JOBS.setdefault(jid, {'status':'queued','created':time.time()}).update(kw)

def get_job(jid):
    with LOCK:
        j = JOBS.get(jid)
        return dict(j) if j else None

def _is_oom(e):
    s = str(e).lower()
    return 'out of memory' in s or 'cuda oom' in s

def render_image(d):
    import torch
    p = (d.get('prompt') or '').strip()
    if p and not p.startswith('masterpiece'):
        p = 'masterpiece, best quality, score_7, ' + p
    neg = d.get('negative') or ('worst quality, low quality, score_1, score_2, score_3, '
                                'artist name, blurry, jpeg artifacts, chromatic aberration')
    if HAS_GPU:
        if pipe is None:
            raise RuntimeError('run cell 3 first')
        gd = (globals().get('GPU_DEFAULTS') or {}).get(globals().get('GPU_MODEL')) or {}
        w = int(d.get('width') or gd.get('w') or 832)
        h = int(d.get('height') or gd.get('h') or 1216)
        steps = int(d.get('steps') or gd.get('steps') or 28)
        seed = int(d.get('seed') or 0)
        kw0 = {}
        if gd.get('guidance') is not None:
            kw0['guidance_scale'] = float(gd['guidance'])
        err = None
        for attempt in range(3):
            try:
                out = pipe(prompt=p, negative_prompt=neg, height=h, width=w,
                           num_inference_steps=steps,
                           generator=torch.Generator('cuda').manual_seed(seed), **kw0)
                img = out.images[0]
                m = float(np.asarray(img.convert('RGB')).mean())
                if m < 6:
                    raise RuntimeError('black image (mean=%.1f) - rerun cell 3' % m)
                buf = io.BytesIO(); img.save(buf, format='PNG')
                return buf.getvalue(), 'png'
            except Exception as e:
                err = e
                if _is_oom(e) and (w > 512 or h > 512):
                    import gc; gc.collect(); torch.cuda.empty_cache()
                    w = max(512, (w*4//5)//8*8); h = max(512, (h*4//5)//8*8)
                    print('[!] OOM -> 降到 %dx%d 重试' % (w, h))
                    continue
                raise
        raise err
    else:
        _pre = str(d.get('cpu_preset') or '').strip()
        if _pre:
            set_cpu_preset(_pre)
        if d.get('force_dtype'):
            CPU['force_dtype'] = str(d['force_dtype']).strip()
        pr = CPU_PRESETS[CPU_PRESET]; ov = load_cpu_pipe(d.get('width'), d.get('height'))
        dim = pr['maxdim']
        w = min(int(d.get('width') or dim), dim); h = min(int(d.get('height') or dim), dim)
        w -= w % 8; h -= h % 8
        kw = dict(prompt=p, height=h, width=w,
                  num_inference_steps=int(d.get('steps') or pr['steps']),
                  guidance_scale=float(d.get('guidance') if d.get('guidance') is not None else pr['guidance']),
                  generator=torch.Generator('cpu').manual_seed(int(d.get('seed') or 0)))
        kw.update(pr.get('extra') or {})
        if pr.get('neg'):
            kw['negative_prompt'] = neg
        _st = {'t0': time.time()}; _jid = d.get('_jid') or ''
        def _prog(pipe_obj, step, timestep, cb_kwargs):
            # diffusers 会对返回值做 .pop('latents', ...)：回调**必须返回 kwargs dict**，
            # 返回 None 直接炸 AttributeError（v3 首版就栽在这，17s/步已证明 fp32 本身可用）。
            # 进度上报本身出错也不能影响出图 → 整段包 try，永远把 kwargs 原样交回去。
            try:
                _n = kw['num_inference_steps']
                if _jid and (step == 0 or step == _n - 1 or step % 2 == 0):
                    set_job(_jid, progress='%d/%d' % (step + 1, _n),
                            elapsed=round(time.time() - _st['t0'], 1))
            except Exception:
                pass
            return cb_kwargs if isinstance(cb_kwargs, dict) else {}
        kw['callback_on_step_end'] = _prog
        img = ov(**kw).images[0]
        m = float(np.asarray(img.convert('RGB')).mean())
        if m < 6:
            raise RuntimeError('black image on CPU (mean=%.1f) -> 换 cpu_preset' % m)
        buf = io.BytesIO(); img.save(buf, format='PNG')
        return buf.getvalue(), 'png'

def worker():
    while True:
        jid, kind, d = TASK_Q.get()
        set_job(jid, status='running', kind=kind)
        t0 = time.time()
        try:
            blob, ext = render_image(d)
            path = os.path.join(OUT_DIR, '%s.%s' % (jid, ext))
            open(path, 'wb').write(blob)
            set_job(jid, status='done', seconds=round(time.time()-t0,1),
                    path=path, bytes=len(blob), ext=ext)
        except Exception as e:
            set_job(jid, status='error', error=('%s: %s' % (type(e).__name__, e))[:300])
        finally:
            TASK_Q.task_done()

def submit(kind, d):
    jid = uuid.uuid4().hex
    d = dict(d); d['_jid'] = jid          # 渲染分支回写进度要用
    set_job(jid, status='queued', kind=kind)
    TASK_Q.put((jid, kind, d))
    return jid

@app.route('/health')
def health():
    return jsonify(ok=True, build=BUILD, gpu=HAS_GPU, gpu_name=GPU_NAME, vram_gb=round(VRAM_GB,1),
                   pending=TASK_Q.qsize(), jobs=len(JOBS), out=OUT_DIR, gpu_model=globals().get('GPU_MODEL'),
                   workdir=WORK, engine='kaggle' if IN_KAGGLE else 'colab',
                   cpu_ready=CPU['pipe'] is not None, cpu_error=CPU['error'],
                   cpu_engine=CPU['engine'], cpu_engine_mode=CPU_ENGINE,
                   cpu_bf16=_cpu_bf16_capable(), cpu_sdpa=CPU_SDPA, cpu_compile=CPU_COMPILE, cpu_slicing=CPU_SLICING,
                   cpu_preset=CPU_PRESET, cpu_presets=list(CPU_PRESETS),
                   avail_ram_gb=round(_avail_ram_gb(), 1), force_dtype=CPU.get('force_dtype'))

@app.route('/cpu_presets')
def cpu_presets_ep():
    return jsonify(current=CPU_PRESET, presets={k: v['model'] for k, v in CPU_PRESETS.items()})

@app.route('/sh', methods=['POST'])
def sh_ep():
    """在 runtime 里执行一条 shell 命令并回 rc/stdout/stderr。

    存在理由：改 cell 4 的代码必须重开 notebook（F5 不重拉 GitHub），而调试 sdcpp
    恰恰需要「看文件在不在 / 跑一下二进制 / 看报错」这类动作。有了这个口，改码之外的
    排查不用再付一次用户重载成本。仅在本人的临时 Colab runtime + 随机隧道下使用。"""
    d = request.get_json(force=True) or {}
    cmd = (d.get('cmd') or '').strip()
    if not cmd:
        return jsonify(error='cmd required'), 400
    try:
        to = float(d.get('timeout') or 300)
    except Exception:
        to = 300.0
    try:
        r = subprocess.run(['bash', '-lc', cmd], capture_output=True, text=True,
                           timeout=to, cwd=d.get('cwd') or SDCPP_DIR)
        return jsonify(ok=r.returncode == 0, rc=r.returncode,
                       out=(r.stdout or '')[-6000:], err=(r.stderr or '')[-4000:])
    except subprocess.TimeoutExpired:
        return jsonify(ok=False, rc=-1, error='timeout after %ss' % to)
    except Exception as e:
        return jsonify(ok=False, rc=-2, error='%s: %s' % (type(e).__name__, str(e))[:200])

@app.route('/generate', methods=['POST'])
def generate():
    d = request.get_json(force=True)
    if not (d.get('prompt') or '').strip():
        return jsonify(error='prompt required'), 400
    kind = d.get('kind') or 'image'
    if kind not in ('image','video'):
        return jsonify(error='kind must be image|video'), 400
    if kind == 'video':
        return jsonify(error='video 未移植到 Kaggle 版（见 cell 0）'), 400
    return jsonify(id=submit(kind, d), queued=TASK_Q.qsize())

@app.route('/batch', methods=['POST'])
def batch():
    d = request.get_json(force=True)
    prompts = d.get('prompts') or []
    if not prompts:
        return jsonify(error='prompts required'), 400
    n = int(d.get('n_per_prompt') or 1)
    kind = d.get('kind') or 'image'
    ids = []
    for pr in prompts:
        for k in range(n):
            dd = dict(d); dd['prompt'] = pr
            if d.get('seed'):
                dd['seed'] = int(d['seed']) + k
            ids.append(submit(kind, dd))
    return jsonify(ids=ids, count=len(ids), queued=TASK_Q.qsize())

@app.route('/status/<jid>')
def status(jid):
    j = get_job(jid)
    if not j:
        return jsonify(error='unknown id'), 404
    return jsonify(j)

@app.route('/image/<jid>')
@app.route('/result/<jid>')
def result(jid):
    j = get_job(jid)
    if not j or j.get('status') != 'done':
        return jsonify(error='not ready'), 404
    mime = 'video/mp4' if j.get('ext') == 'mp4' else 'image/png'
    return send_file(j['path'], mimetype=mime)

@app.route('/upload', methods=['POST'])
def upload():
    f = request.files.get('file')
    if not f:
        return jsonify(error='file field required'), 400
    name = re.sub(r'[^A-Za-z0-9_.-]', '_', f.filename or 'ref.png')
    path = os.path.join(OUT_DIR, 'up_' + name)
    f.save(path)
    return jsonify(ok=True, path=path)

@app.route('/flush')
def flush():
    flush_to_drive()
    return jsonify(ok=True, drive=DRIVE_OK)

# 幂等启动：重跑本格不再报 Address already in use（旧 Flask 活着就复用）
try:
    urllib.request.urlopen('http://127.0.0.1:7860/health', timeout=3)
    print('[i] Flask 已在 7860 运行，复用旧实例（模型走全局 pipe，仍然生效）')
except Exception:
    threading.Thread(target=worker, daemon=True).start()
    threading.Thread(target=lambda: app.run(host='0.0.0.0', port=7860, debug=False, use_reloader=False),
                     daemon=True).start()

# 半截下载的二进制会 EACCES：用 --version 实测可用性，坏就重下；chmod 无条件执行
BIN = os.path.join(WORK, 'cloudflared')   # Kaggle 的 /usr/local/bin 不一定可写
def _cf_ok():
    try:
        return subprocess.run([BIN,'--version'], capture_output=True, timeout=20).returncode == 0
    except Exception:
        return False
if not _cf_ok():
    subprocess.run(['rm','-f',BIN])
    subprocess.run(['wget','-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O',BIN], check=True)
os.chmod(BIN, 0o755)
subprocess.run(['pkill','-f','cloudflared tunnel'], capture_output=True); time.sleep(1)
proc = subprocess.Popen([BIN,'tunnel','--url','http://127.0.0.1:7860','--no-autoupdate'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if 'trycloudflare.com' in line:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            URL = m.group(0); break

print('=' * 64)
print('PUBLIC URL:', URL)
print('=' * 64)
print("存到本机:  echo '%s' > ~/.dsh/.colab-url" % URL)
print('本机 CLI:  scripts/colab_gen.sh "prompt"')

In [ ]:
#@title 5. 自测一张（640x960@22，白衬衫同款配置）
import requests, time, io
from PIL import Image
r = requests.post(URL + '/generate', json={
    'prompt': '1girl, solo, silver long hair, blue eyes, white shirt, soft window light, detailed skin',
    'width': 832, 'height': 1216, 'steps': 28, 'seed': 4242})
jid = r.json()['id']; print('job', jid)
t0 = time.time()
s = {}
while time.time() - t0 < 600:
    s = requests.get(URL + '/status/' + jid, timeout=30).json()
    st = s.get('status')
    if st in ('done', 'error'):
        print(st, s.get('seconds',''), (s.get('error') or '')[:150]); break
    time.sleep(5)
if s.get('status') == 'done':
    img = Image.open(io.BytesIO(requests.get(URL + '/result/' + jid, timeout=120).content))
    print('出图:', img.size, '%.0fs' % s.get('seconds', 0), '| model=%s' % globals().get('GPU_MODEL'))
    display(img)

In [ ]:
#@title 6. 批量出图（改 PROMPTS 一次提交，服务端排队串行出）
PROMPTS = [
    '1girl, solo, silver hair, standing in rain, city night, neon reflections',
    '1girl, solo, red kimono, cherry blossoms, sunlight, bokeh',
    '1girl, solo, black hair, reading in library, warm lamp light',
]
N_PER_PROMPT = 2
W, H, STEPS, SEED = 832, 1216, 28, 1000

import requests, time
r = requests.post(URL + '/batch', json={'prompts': PROMPTS, 'n_per_prompt': N_PER_PROMPT,
                                        'width': W, 'height': H, 'steps': STEPS, 'seed': SEED})
ids = r.json()['ids']
per = 30 if HAS_GPU else 300
print('[+] 入队 %d 张，预计 ~%.0f 分钟' % (len(ids), len(ids)*per/60))
t0 = time.time()
while time.time() - t0 < len(ids)*per*4 + 600:
    st = [get_job(i) or {} for i in ids]
    d = sum(1 for s in st if s.get('status') == 'done')
    e = sum(1 for s in st if s.get('status') == 'error')
    print('  %d+%de/%d  %.1fmin' % (d, e, len(ids), (time.time()-t0)/60))
    if d + e >= len(ids):
        break
    time.sleep(10)
flush_to_drive()
print('[!] /content 会话结束清空，产出已搬 Drive')

In [ ]:
#@title 8. CPU 精度基准：bf16 / fp32 / fp16 单步计时（决定方案一值不值）
# 用最小的 sd_turbo（fp32 才 5.2GB，三档都装得下不会 OOM）实测每档单步耗时。
# 直接回答：bf16 在这台 Colab CPU 上到底比 fp32 快还是慢（取决于有无 avx512_bf16/amx_bf16）。
import time, gc
from diffusers import StableDiffusionPipeline
_f = globals().get('CPU_FLAGS', set())
print('bf16 硬件探测:', _cpu_bf16_capable(), '| 命中位:',
      ' '.join(sorted(x for x in _f if 'avx512' in x or 'amx' in x)) or '无')
_res = {}
for dt in ('bfloat16', 'float32', 'float16'):
    gc.collect()
    try:
        kw = {'use_safetensors': True}
        if dt in ('float16', 'bfloat16'):
            kw['variant'] = 'fp16'          # sd-turbo 有 fp16 存储，bf16 由它转
        p = StableDiffusionPipeline.from_pretrained('stabilityai/sd-turbo', dtype=getattr(torch, dt), **kw)
        p.set_progress_bar_config(disable=True)
        try:
            p.unet.set_attention_backend('native')
        except Exception:
            pass
        p.to('cpu')
        g = lambda s: torch.Generator('cpu').manual_seed(s)
        p('a girl', num_inference_steps=1, guidance_scale=0.0, height=512, width=512, generator=g(0))  # 预热
        t0 = time.time()
        p('a girl', num_inference_steps=2, guidance_scale=0.0, height=512, width=512, generator=g(1))
        _res[dt] = (time.time() - t0) / 2
        print('  %-9s : ~%.1f s/步' % (dt, _res[dt]))
        del p; gc.collect()
    except Exception as e:
        print('  %-9s : 失败 %s: %s' % (dt, type(e).__name__, str(e)[:120]))
        gc.collect()
if _res.get('bfloat16') and _res.get('float32'):
    _r = _res['float32'] / _res['bfloat16']
    print('\n[i] bf16 vs fp32 加速比 = %.2fx' % _r)
    if _r > 1.3:
        print('    → bf16 值得用：SDXL 走 bfloat16 既装得下(7.3GB)又快，方案一成立')
    else:
        print('    → 这台 CPU 无 bf16 加速。SDXL 在 Colab CPU 上无解（fp32 装不下 / fp16 慢一个量级）')
        print('      正解：等 GPU 配额，或走方案二 sdcpp / 方案三 Modal')

## Kaggle 跑法与排错

**流程**：Settings 选 `GPU T4 x2` + `Internet On` → 交互式 Run **1 → 2 → 3 → 4** → 复制 `PUBLIC URL`
→ 本机 `echo '<url>' > ~/.dsh/.colab-url`（或 `~/.dsh/.kaggle-url`，等价）→ `scripts/colab_gen.sh "prompt"`。

**不能 "Save & Run All"**：URL 是运行期输出；kernel 空闲自动停、隧道随即断。定位是**手动出图环境**。

**常见问题**
1. cell 1 依赖报错 → 贴原文；先 cell 3 改 `GPU_MODEL='sd_turbo'` 跑 512×768@4 验链路
2. 黑图（mean<6）→ 改 `GPU_MODEL='nova_lcm'` 或降分辨率
3. cell 4 无 URL → Internet 未开（wget 失败）或 7860 被占
4. `/health` 的 `gpu:false` → Accelerator 未生效，重启 kernel 再跑 cell 1
5. 产物：`/kaggle/working/out/<jid>.png`，或 `/result/<jid>` 拉回

**实测已确认**：animagine fp16 832×1216@28 = 22.1s/张，mean 161.8；tunnel 出网正常（iad21）。
**待补**：animagine 加载秒数、sd-turbo 512×768@4 全项、剩余 GPU 小时。
